In [1]:
from math_datasets.datasets import Dataset, GSM8K, SVAMP
from math_datasets.evaluator import evaluate_all, evaluate_detail
from pathlib import Path
from math_datasets.generators.react import ReactGenerate
import os

/opt/miniconda3/envs/MA312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MPS (Metal Performance Shaders) is available! Using Apple Silicon GPU.


In [2]:
model = "qwen3:0.6b"
SAVE_DIR = Path(os.getcwd())
def get_ollama_model_name_identifer(model_name: str) -> str:
    return "ollama/" + model_name

model_identifier_run_name = get_ollama_model_name_identifer(model)

In [3]:
df_detail = evaluate_detail(model_identifier_run_name, SVAMP, SAVE_DIR.as_posix(), use_transformated_answers=False, additional_metrics=ReactGenerate)

In [4]:
df_detail.head()

,ID,Body,Question,Equation,Answer,Type,question_concat,model_history,response,is_correct,input_tokens,output_tokens,total_tokens,format_correct
0,chal-736,Winter is almost here and most animals are mig...,How many more bird families flew away to afric...,( 62.0 - 35.0 ),27,Subtraction,Winter is almost here and most animals are mig...,"[{'type': 'human', 'data': {'content': 'Winter...","<think>\nOkay, let's see. The problem says the...",True,227,299,526,True
1,chal-162,Paige raised 7 goldfish and 12 catfish in the ...,How many fishes disappeared?,( ( 7.0 + 12.0 ) - 15.0 ),4,Subtraction,Paige raised 7 goldfish and 12 catfish in the ...,"[{'type': 'human', 'data': {'content': 'Paige ...","<think>\nOkay, let's see. The problem says tha...",True,201,288,489,True
2,chal-349,Marco and his dad went strawberry picking. Tog...,How much did his dad's strawberries weigh now?,( ( 22.0 - 36.0 ) + 30.0 ),16,Addition,Marco and his dad went strawberry picking. Tog...,"[{'type': 'human', 'data': {'content': 'Marco ...","<think>\nOkay, let's see. The problem says Mar...",True,220,330,550,True
3,chal-390,Debby bought 200 water bottles and 256 soda bo...,How many days would the soda bottles last?,( 256.0 / 4.0 ),64,Common-Division,Debby bought 200 water bottles and 256 soda bo...,"[{'type': 'human', 'data': {'content': 'Debby ...","<think>\nOkay, let's see. Debby bought 200 wat...",True,213,294,507,True
4,chal-781,There were 106 dollars in Olivia's wallet. Aft...,How much did she spend at the supermarket?,( ( 106.0 - 26.0 ) - 49.0 ),31,Subtraction,There were 106 dollars in Olivia's wallet. Aft...,"[{'type': 'human', 'data': {'content': 'There ...","<think>\nOkay, let's see. The problem says Oli...",True,212,646,858,True


## Correct Analysis

In [12]:
# number correct
df_detail["is_correct"].value_counts(normalize=True)

is_correct
True     0.83
False    0.17
Name: proportion, dtype: float64

In [13]:
# True: in x % of answers, when the format was correct it provided the right answer.
df_detail[df_detail["format_correct"] == True]["is_correct"].value_counts(normalize=True)

is_correct
True     0.83
False    0.17
Name: proportion, dtype: float64

## Format Analysis

In [14]:
df_detail["format_correct"].value_counts(normalize=True)

format_correct
True    1.0
Name: proportion, dtype: float64

In [16]:
# Format Correct Score where the end result is false
## If the score for False is high this means that if the answer is incorrect, then it is most likely due to incorrect format
df_detail[df_detail["is_correct"] == False]["format_correct"].value_counts(normalize=True)

format_correct
True    1.0
Name: proportion, dtype: float64

## Token Usage

In [17]:
df_detail["input_tokens"].mean(), df_detail["output_tokens"].mean(), df_detail["total_tokens"].mean()

(np.float64(231.27), np.float64(470.81), np.float64(702.08))

In [18]:
df_format_correct = df_detail[df_detail["format_correct"] == True]
df_correct = df_format_correct[df_format_correct["is_correct"] == True]

df_correct["input_tokens"].mean(), df_correct["output_tokens"].mean(), df_correct["total_tokens"].mean()

(np.float64(235.6987951807229),
 np.float64(376.67469879518075),
 np.float64(612.3734939759037))

In [19]:
df_wrong = df_format_correct[df_format_correct["is_correct"] == False]
df_wrong["input_tokens"].mean(), df_wrong["output_tokens"].mean(), df_wrong["total_tokens"].mean()

(np.float64(209.64705882352942),
 np.float64(930.4117647058823),
 np.float64(1140.0588235294117))